# Parte Comum: K-Means, Passada 2 (k fixo)

Dataset: Obesity (UCI 544), arquivo `data/parte_comum/obesity.csv`.

## Etapa 1: Padronização das variáveis

In [ ]:
# [IA] Claude Code – importacao das bibliotecas
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
# [IA] Claude Code – leitura do dataset (caminho relativo a pasta do notebook)
CAMINHO_DADOS = Path("../../data/parte_comum/obesity.csv")

df = pd.read_csv(CAMINHO_DADOS)
print(df.shape)
df.head()

In [ ]:
# [IA] Claude Code – separar variaveis de entrada (X) e classe real (y)
# A classe real NAO entra no K-Means; fica guardada para o mapeamento e as metricas.
COLUNA_CLASSE = "NObeyesdad"

y = df[COLUNA_CLASSE].copy()
X = df.drop(columns=[COLUNA_CLASSE]).copy()

print("Classes reais:", y.nunique())
print(y.value_counts())

In [ ]:
# [IA] Claude Code – variaveis numericas usadas como entrada do K-Means (lista explicita)
# O K-Means agrupa por distancia euclidiana, que so faz sentido para variaveis numericas.
# As categoricas (Gender, family_history_with_overweight, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS)
# continuam sendo analisadas na EDA; so nao entram no agrupamento.
# A Passada 1 (kmeans_passada1.ipynb) deve usar exatamente esta mesma lista.
VARIAVEIS_KMEANS = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]

X = X[VARIAVEIS_KMEANS]

# Conferencia: todas as variaveis escolhidas sao numericas
assert all(pd.api.types.is_numeric_dtype(X[c]) for c in VARIAVEIS_KMEANS), "Alguma variavel nao e numerica"
X.head()

In [ ]:
# [IA] Claude Code – padronizacao (z-score): media 0 e desvio padrao 1 em cada variavel
scaler = StandardScaler()
X_padronizado = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index,
)
X_padronizado.head()

In [ ]:
# [IA] Claude Code – conferir a padronizacao (media ~ 0 e desvio ~ 1)
conferencia = pd.DataFrame({
    "media_antes": X.mean(),
    "desvio_antes": X.std(ddof=0),
    "media_depois": X_padronizado.mean().round(6),
    "desvio_depois": X_padronizado.std(ddof=0).round(6),
})
conferencia

### Por que padronizar antes do K-Means?

_(texto do grupo)_

## Etapa 2: K-Means com k = número de classes reais

In [ ]:
# [IA] Claude Code – k fixo no numero de classes reais do dataset
from sklearn.cluster import KMeans

K_FIXO = y.nunique()
print("k fixo =", K_FIXO)

In [ ]:
# [IA] Claude Code – executar o K-Means sobre os dados padronizados
# random_state fixo para o resultado ser reproduzivel; n_init=10 roda 10 inicializacoes e fica com a melhor
kmeans_fixo = KMeans(n_clusters=K_FIXO, n_init=10, random_state=42)
clusters_fixo = kmeans_fixo.fit_predict(X_padronizado)

print("Inercia (soma das distancias ao quadrado):", round(kmeans_fixo.inertia_, 2))

In [ ]:
# [IA] Claude Code – guardar o cluster de cada linha junto com a classe real (usado no mapeamento e nas metricas)
resultado_fixo = pd.DataFrame({
    "cluster": clusters_fixo,
    "classe_real": y.values,
}, index=X.index)

# Quantidade de pontos em cada cluster
resultado_fixo["cluster"].value_counts().sort_index().rename("qtd_pontos").to_frame()

In [ ]:
# [IA] Claude Code – centroides de cada cluster na escala original das variaveis
centroides_fixo = pd.DataFrame(
    scaler.inverse_transform(kmeans_fixo.cluster_centers_),
    columns=X.columns,
).round(2)
centroides_fixo.index.name = "cluster"
centroides_fixo